<a href="https://colab.research.google.com/github/ahmedgx1010/Chess-Engine/blob/main/Chess_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 0: setup.

### installing libraries.

In [2]:
!pip install chess
!apt-get -qq install zstd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=a8391b883661629983bdbee2ff981d3d81d277e7e638ca2a206af8c129bd3b2a
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess
Selecting previously unselected package zstd.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


### Imports:
All the libraries used in this project.

In [3]:
import chess
import json
import random
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

# Part 1: Search engine

### material evaluation:
Scores a position by counting material

In [4]:
piece_values={
    chess.PAWN:1,
    chess.KNIGHT:3,
    chess.BISHOP:3,
    chess.ROOK:5,
    chess.QUEEN:9,
    chess.KING:0,
}

def evaluate(board):
  if board.is_checkmate():
    if board.turn == chess.WHITE:
      return -1000
    else:
      return 1000
  if board.is_game_over():
    return 0

  score = 0
  for square, piece in board.piece_map().items():
    value = piece_values[piece.piece_type]
    if piece.color == chess.WHITE:
      score += value
    else:
      score -= value
  return score

### Minimax:
Look 'depth' moves ahead, 'best_move' returns the move with the best score

In [5]:
def minimax(board, depth, maximizing):
  if depth == 0 or board.is_game_over():
    return evaluate(board)

  if maximizing:
    best = -float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = minimax(board, depth - 1, False)
      board.pop()
      best = max(best,score)
    return best
  else:
    best = float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = minimax(board, depth - 1, True)
      board.pop()
      best = min(best, score)
    return best

def best_move(board, depth):
  maximizing = board.turn == chess.WHITE
  best_score = -float("inf") if maximizing else float("inf")
  chosen = None

  for move in board.legal_moves:
    board.push(move)
    score = minimax(board, depth - 1, not maximizing)
    board.pop()

    if maximizing and score > best_score:
      best_score, chosen = score, move
    elif not maximizing and score < best_score:
      best_score, chosen = score, move

  return chosen

### Alpha-beta pruning:
Same result as minimax Same result as minimax, but skips branches that can’t change the final decision, so it’s much faster, so it's much faster. `eval_fn` chooses which evaluation function to use when playing.

In [6]:
def alphabeta(board, depth, alpha, beta, maximizing, eval_fn=evaluate):
    if depth == 0 or board.is_game_over():
        return eval_fn(board)

    if maximizing:
      best = -float("inf")
      for move in board.legal_moves:
        board.push(move)
        score = alphabeta(board, depth - 1, alpha, beta, False, eval_fn)
        board.pop()
        best = max(best, score)
        alpha = max(alpha, best)
        if alpha >= beta:
          break
      return best
    else:
      best = float("inf")
      for move in board.legal_moves:
        board.push(move)
        score = alphabeta(board, depth - 1, alpha, beta, True, eval_fn)
        board.pop()
        best = min(best, score)
        beta = min(beta, best)
        if alpha >= beta:
          break
      return best

def best_move_alphabeta(board, depth, eval_fn=evaluate):
  maximizing = board.turn == chess.WHITE
  best_score = -float("inf") if maximizing else float("inf")
  chosen = None

  for move in board.legal_moves:
    board.push(move)
    score = alphabeta(board, depth - 1, -float("inf"), float("inf"), not maximizing, eval_fn)
    board.pop()

    if maximizing and score > best_score:
      best_score, chosen = score, move
    elif not maximizing and score < best_score:
      best_score, chosen = score, move

  return chosen

# Part 2: Training the ML evaluation

### Download the data:
Downloads positions scored by Stockfish from the Lichess evaluation database. I used 500,000; use fewer or more depened on you memory

### Read positions and scores:
Keeps each position (FEN) and Stockfish's score in centipawns. Positions with a forced mate are skipped.

In [8]:
fens = []
scores = []

with open("evals.jsonl") as f:
  for line in f:
    data = json.loads(line)
    best = data["evals"][0]["pvs"][0]
    if "cp" in best:
      fens.append(data["fen"])
      scores.append(best["cp"])

print(len(fens), "positions loaded")

406366 positions loaded


### Board to features:
Turns a board into 768 zeros and ones: 12 piece types × 64 squares. 1 = that piece is on that square.

In [9]:
def board_to_features(board):
  features = np.zeros(768, dtype=np.uint8)
  for square, piece in board.piece_map().items():
    index = (piece.piece_type - 1) * 64 + square
    if piece.color == chess.BLACK:
      index += 6 * 64
    features[index] = 1
  return features

### Build the dataset:
`X` = features for every position (one row each). `y` = Stockfish scores.

In [10]:
N = len(fens)
X = np.zeros((N, 768), dtype=np.uint8)
y = np.array(scores, dtype=np.float32)

for i in range(N):
  X[i] = board_to_features(chess.Board(fens[i]))
  if i % 100000 == 0:
    print(i, "done")

del fens, scores

0 done
100000 done
200000 done
300000 done
400000 done


### Clip scores and split the data:
Caps scores at ±1000 so extreme positions don't affect training. 80% of positions are used for training, 20% are kept for testing.

In [11]:
y_clipped = np.clip(y, -1000, 1000)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clipped, test_size=0.2, random_state=42)

### Linear regression and baseline:
Linear regression learns one weight per piece-on-square. The baseline always guesses the average score, to check the model actually learned something.

In [12]:
model = LinearRegression()
model.fit(X_train, y_train)

model_error = mean_absolute_error(y_test, model.predict(X_test))

baseline = np.full(len(y_test), y_train.mean())
baseline_error = mean_absolute_error(y_test, baseline)

### Neural network (MLP):
Two hidden layers (128 and 64 neurons), so it can learn combinations of pieces. Trained on scores in pawns; early stopping prevents overfitting.

In [13]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128,64),
    max_iter=20,
    early_stopping=True,
    random_state=42,
    verbose=True
)

mlp.fit(X_train, y_train / 100);

Iteration 1, loss = 2.88217876
Validation score: 0.443735
Iteration 2, loss = 1.96157149
Validation score: 0.512697
Iteration 3, loss = 1.59737599
Validation score: 0.541798
Iteration 4, loss = 1.39516930
Validation score: 0.556884
Iteration 5, loss = 1.25292691
Validation score: 0.568793
Iteration 6, loss = 1.14927656
Validation score: 0.566558
Iteration 7, loss = 1.06988622
Validation score: 0.573829
Iteration 8, loss = 1.00575891
Validation score: 0.559280
Iteration 9, loss = 0.95115828
Validation score: 0.563951
Iteration 10, loss = 0.90843384
Validation score: 0.573339
Iteration 11, loss = 0.86843430
Validation score: 0.569900
Iteration 12, loss = 0.83477691
Validation score: 0.568538
Iteration 13, loss = 0.80401332
Validation score: 0.574633
Iteration 14, loss = 0.77712097
Validation score: 0.556784
Iteration 15, loss = 0.75247750
Validation score: 0.565096
Iteration 16, loss = 0.73224336
Validation score: 0.560724
Iteration 17, loss = 0.71564638
Validation score: 0.556847
Iterat

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor(early_stopping=True, hidden_layer_sizes=(128, 64), max_iter=20,
             random_state=42, verbose=True)

### Compare the models:
Average error on the test positions between the models, in centipawns. Lower is better.

In [14]:
mlp_error = mean_absolute_error(y_test, mlp.predict(X_test) * 100)

print("MLP error:    ", round(mlp_error), "centipawns")
print("Linear error:    ", round(model_error), "centipawns")
print("Baseline error:    ", round(baseline_error), "centipawns")

MLP error:     117 centipawns
Linear error:     158 centipawns
Baseline error:     171 centipawns


# Part 3: Learned engines

### Evaluation functions:
Use the trained models to score positions, in pawns. The hybrid adds the MLP's score to exact material counting.

In [15]:
def evaluate_lr(board):
    if board.is_game_over():
        return evaluate(board)
    features = board_to_features(board)
    return model.predict(features.reshape(1, -1))[0] / 100

def evaluate_ml(board):
  if board.is_game_over():
    return evaluate(board)
  features = board_to_features(board)
  return mlp.predict(features.reshape(1, -1))[0]

def evaluate_hybrid(board):
    if board.is_game_over():
        return evaluate(board)
    return evaluate(board) + evaluate_ml(board)

### Best-move functions for the learned engines:
Using the Alpha-Beta search for faster result

In [16]:
def best_move_lr(board, depth):
    return best_move_alphabeta(board, depth, evaluate_lr)

def best_move_ml(board, depth):
    return best_move_alphabeta(board, depth, evaluate_ml)

def best_move_hybrid(board, depth):
    return best_move_alphabeta(board, depth, evaluate_hybrid)

# Part 4: Playing

### Engine list:
All available engines, by number.

In [17]:
engines = {
    1: ("minimax", best_move),
    2: ("material", best_move_alphabeta),
    3: ("Linear", best_move_lr),
    4: ("MLP", best_move_ml),
    5: ("Hybrid", best_move_hybrid),
}
settings = {"engine": 2, "depth": 3}

### Game board:
The board for games against the engine. Rerun only to get a completely fresh board.

In [18]:
game = chess.Board()

### Play against an engine:
Start with `new_game(engine, depth)`, then play with `my_move("your move")` in a new cell. You play white.

In [19]:
def show_engines():
  for number, (name, _) in engines.items():
    print(f"{number}: {name}")

def new_game(engine=2, depth=3):
  if engine not in engines:
    print("Unknown engine, Choose a number:")
    show_engines()
    return
  game.reset()
  settings["engine"] = engine
  settings["depth"] = depth
  name, _ = engines[engine]
  print(f"New game: you are white, engine is {name} at depth {depth}")
  display(game)

def my_move(move_text):
  try:
    game.push_san(move_text)
  except ValueError:
    legal = [game.san(m) for m in game.legal_moves]
    print(f"'{move_text}' is illegal. legal moves: {', '.join(legal)}")
    return

  if not game.is_game_over():
    name, engine = engines[settings["engine"]]
    reply = engine(game, settings["depth"])
    print(f"Engine ({name}) played:", game.san(reply))
    game.push(reply)

  display(game)
  if game.is_game_over():
    print("game over:", game.result())

### Engine vs. engine:
Each game starts with 4 random moves, so each game begins from a different position. Each opening is played twice with colors swapped. Win = 1, draw = 0.5.

In [20]:
def play_match_game(white_engine, black_engine, depth=2 , random_moves=4, max_moves =150):
  board = chess.Board()

  for _ in range(random_moves):
    move = random.choice(list(board.legal_moves))
    board.push(move)

  while not board.is_game_over() and board.ply() < max_moves:
    if board.turn == chess.WHITE:
      _, engine = engines[white_engine]
    else:
      _, engine = engines[black_engine]
    board.push(engine(board, depth))

  if board.is_game_over():
      return board.result()
  print("  (stopped at move limit)")
  return "1/2-1/2"

def match(engine_a, engine_b, games=10, depth=2):
  score_a = 0

  for i in range(games):
    random.seed(i // 2)

    if i % 2 == 0:
      result = play_match_game(engine_a, engine_b, depth)
      if result == "1-0":
        score_a += 1
    else:
      result = play_match_game(engine_b, engine_a, depth)
      if result == "0-1":
        score_a += 1

    if result == "1/2-1/2":
      score_a += 0.5

    print(f"Game {i + 1}: {result}")

  name_a, _ = engines[engine_a]
  name_b, _ = engines[engine_b]
  print(f"\n{name_a} scored {score_a} out of {games} against {name_b}")

# Part 5: Experiments

### Speed: minimax vs. alpha-beta:
Same position and depth. Both should find the same move; alpha-beta should be much faster.

In [21]:
test_board = chess.Board()
for m in ["e4", "e5", "Qh5", "Nc6"]:
  test_board.push_san(m)

start = time.time()
print("Minimax:  ", best_move(test_board, 3), round(time.time() - start, 2), "sec")

start = time.time()
print("Alpha-beta:  ", best_move_alphabeta(test_board, 3), round(time.time() - start, 2), "sec")

Minimax:   h5f5 3.44 sec
Alpha-beta:   h5f5 0.26 sec


### Piece values learned by the linear model:
Average weight per piece type, in centipawns. The model learned these from data alone.

In [22]:
weights = model.coef_.reshape(12,64)

names = ["White pawn", "White knight", "White bishop", "White rook", "White queen", "White king",
         "Black pawn", "Black knight", "Black bishop", "Black rook", "Black queen", "Black king"]
for i, name in enumerate(names):
  row = weights[i]
  if "pawn" in name:
    row = row [8:56]
  print(f"{name:13} {row.mean():7.0f}")

White pawn        110
White knight      158
White bishop      204
White rook        247
White queen       460
White king          0
Black pawn       -106
Black knight     -160
Black bishop     -164
Black rook       -267
Black queen      -448
Black king         -0


### The missing-queen test:
Starting position without white's queen. Shows the MLP's blind spot: it barely notices lost material.

In [23]:
b = chess.Board()
b.remove_piece_at(chess.D1)

print("Material:", evaluate(b))
print("MLP:     ", evaluate_ml(b))

Material: -9
MLP:      -0.37996267086650853


### Match: MLP vs. material:
20 games at depth 2.

In [24]:
match(4, 2, games=20)

Game 1: 0-1
  (stopped at move limit)
Game 2: 1/2-1/2
  (stopped at move limit)
Game 3: 1/2-1/2
  (stopped at move limit)
Game 4: 1/2-1/2
  (stopped at move limit)
Game 5: 1/2-1/2
Game 6: 1/2-1/2
  (stopped at move limit)
Game 7: 1/2-1/2
  (stopped at move limit)
Game 8: 1/2-1/2
  (stopped at move limit)
Game 9: 1/2-1/2
  (stopped at move limit)
Game 10: 1/2-1/2
Game 11: 0-1
  (stopped at move limit)
Game 12: 1/2-1/2
Game 13: 0-1
Game 14: 1-0
Game 15: 0-1
  (stopped at move limit)
Game 16: 1/2-1/2
  (stopped at move limit)
Game 17: 1/2-1/2
  (stopped at move limit)
Game 18: 1/2-1/2
  (stopped at move limit)
Game 19: 1/2-1/2
  (stopped at move limit)
Game 20: 1/2-1/2

MLP scored 7.5 out of 20 against alphabeta


### Match: Hybrid vs. material:
20 games at depth 2.

In [25]:
match(5, 2, games=20)

  (stopped at move limit)
Game 1: 1/2-1/2
  (stopped at move limit)
Game 2: 1/2-1/2
  (stopped at move limit)
Game 3: 1/2-1/2
  (stopped at move limit)
Game 4: 1/2-1/2
  (stopped at move limit)
Game 5: 1/2-1/2
Game 6: 0-1
Game 7: 1-0
Game 8: 1-0
Game 9: 1-0
  (stopped at move limit)
Game 10: 1/2-1/2
Game 11: 1-0
Game 12: 1/2-1/2
  (stopped at move limit)
Game 13: 1/2-1/2
Game 14: 0-1
  (stopped at move limit)
Game 15: 1/2-1/2
  (stopped at move limit)
Game 16: 1/2-1/2
  (stopped at move limit)
Game 17: 1/2-1/2
Game 18: 1-0
Game 19: 1-0
Game 20: 0-1

Hybrid scored 12.5 out of 20 against alphabeta
